# JSON Lines 形式でデータを読み書きする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/15_data_io/jsonl_io.ipynb)

NLP クックブック 15-06 ／ データの収集と入出力

1 行に 1 件の JSON を書く JSONL 形式で、データセットや実験結果を保存・読み込みしたい。

In [1]:
import json

records = [
    {"id": 1, "text": "今日は良い天気です。", "label": "positive", "tokens": ["今日", "は", "良い", "天気", "です", "。"]},
    {"id": 2, "text": "改行\nや\"引用符\"、絵文字😀を含む文", "label": "neutral", "tokens": []},
]

with open("data.jsonl", "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")   # ensure_ascii=False で日本語をそのまま書く

print(open("data.jsonl", encoding="utf-8").read())

with open("data.jsonl", encoding="utf-8") as f:
    loaded = [json.loads(line) for line in f if line.strip()]
print(loaded == records)

{"id": 1, "text": "今日は良い天気です。", "label": "positive", "tokens": ["今日", "は", "良い", "天気", "です", "。"]}
{"id": 2, "text": "改行\nや\"引用符\"、絵文字😀を含む文", "label": "neutral", "tokens": []}

True


In [2]:
import pandas as pd

df = pd.read_json("data.jsonl", lines=True)                    # pandas で読む
df.to_json("out.jsonl", orient="records", lines=True, force_ascii=False)   # pandas で書く
print(df)

# 途中で落ちても結果を失わないよう、1 件処理するごとに追記する (API 呼び出しや長時間の推論で重要)
import os

done = {json.loads(l)["id"] for l in open("results.jsonl", encoding="utf-8")} if os.path.exists("results.jsonl") else set()
with open("results.jsonl", "a", encoding="utf-8") as f:
    for r in records:
        if r["id"] in done:
            continue                                            # 再実行時は済んだものを飛ばす
        f.write(json.dumps({"id": r["id"], "length": len(r["text"])}, ensure_ascii=False) + "\n")
        f.flush()
print(open("results.jsonl", encoding="utf-8").read())

   id                 text     label                  tokens
0   1           今日は良い天気です。  positive  [今日, は, 良い, 天気, です, 。]
1   2  改行\nや"引用符"、絵文字😀を含む文   neutral                      []
{"id": 1, "length": 10}
{"id": 2, "length": 18}



- CSV と違って、改行・カンマ・引用符を含むテキストやリスト・入れ子の構造を安全に扱える。NLP のデータ交換では事実上の標準。
- 1 行ずつ読めるので、メモリに載らない巨大なファイルも処理できる。`.jsonl.gz` のまま `gzip.open(path, "rt")` で読める。